# NDWI Evaluation — WorldView-3 Test Chips

**Two evaluations performed:**
- **Honest baseline (t=0.35):** threshold selected on 51 development scenes, applied to test set → micro F1=0.523
- **Upper bound (t=0.70):** threshold optimised directly on test labels → micro F1=0.601

All results saved. **Do not re-run** — results are already saved to `results/ndwi/ndwi_dev_threshold.json`.

In [ ]:
# ── Cell 1: Setup ──
import os
import numpy as np
import rasterio
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score
import json
from tqdm import tqdm

# Paths — override by setting the BASE_DIR environment variable
BASE_DIR   = Path(os.environ.get(
    'BASE_DIR',
    '/mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask'
))
CHIPS_DIR  = BASE_DIR / 'images/chips_images/images'
MASKS_DIR  = BASE_DIR / 'images/chips_masks/masks'
NDWI_DIR   = BASE_DIR / 'results/ndwi'
OUTPUT_DIR = BASE_DIR / 'results/ndwi_chips'

with open(BASE_DIR / 'results/unet_training/test_chips.json') as f:
    test_chip_names = set(json.load(f))

all_chips  = sorted(CHIPS_DIR.glob('*.tif'))
test_chips = [c for c in all_chips if c.name in test_chip_names]
dev_chips  = [c for c in all_chips if c.name not in test_chip_names]

# Band indices (1-based for rasterio)
GREEN_BAND = 3   # band 3 = Green
NIR_BAND   = 7   # band 7 = NIR1
print(f'Test chips: {len(test_chips)}, Dev chips: {len(dev_chips)}')
print(f'NDWI = (Green band{GREEN_BAND} - NIR band{NIR_BAND}) / (Green + NIR)')


Test chips: 4788, Dev chips: 12657
NDWI = (Green band3 - NIR band7) / (Green + NIR)

## Evaluation 1: Test-Label-Optimised Threshold (Upper Bound)

Threshold swept directly on test chips and labels. Best: **t=0.70, micro F1=0.601**.
Because the threshold was selected using test labels, this is an optimised upper bound — not achievable in deployment.

In [ ]:
# ── Cell 2: Test-optimised threshold sweep (upper bound) ──
# NOTE: Already completed. Results saved to results/ndwi_chips/ndwi_micro_sweep.csv
# Threshold sweep: -0.3 to 0.8 across 12 values on 4,788 test chips.
# Best micro threshold: t=0.70 (F1=0.601, P=0.571, R=0.635)
# This result represents an optimised upper bound — threshold selected using test labels.

micro_sweep_csv = OUTPUT_DIR / 'ndwi_micro_sweep.csv'
if micro_sweep_csv.exists():
    micro_sweep = pd.read_csv(micro_sweep_csv)
    best_row = micro_sweep.loc[micro_sweep['micro_f1'].idxmax()]
    print(f'Test-optimised threshold: t={best_row["threshold"]}')
    print(f'Micro F1:  {best_row["micro_f1"]:.4f}')
    print(f'Precision: {best_row["micro_precision"]:.4f}')
    print(f'Recall:    {best_row["micro_recall"]:.4f}')
    print(micro_sweep[['threshold','micro_f1','micro_precision','micro_recall']].to_string(index=False))
else:
    print('Results file not found — run ndwi sweep first')

Test-optimised threshold: t=0.7
Micro F1:  0.6014
Precision: 0.5709
Recall:    0.6354
 threshold  micro_f1  micro_precision  micro_recall
      -0.3    0.0886           0.0464        0.9958
      -0.2    0.1242           0.0662        0.9904
      -0.1    0.1778           0.0978        0.9807
       0.0    0.2390           0.1364        0.9648
       0.1    0.3911           0.2469        0.9409
       0.2    0.4485           0.2974        0.9115
       0.3    0.5018           0.3512        0.8786
       0.4    0.5413           0.4004        0.8354
       0.5    0.5679           0.4484        0.7742
       0.6    0.5879           0.5043        0.7048
       0.7    0.6014           0.5709        0.6354
       0.8    0.6006           0.6494        0.5587

## Evaluation 2: Development-Data Threshold (Honest Baseline)

Threshold selected on 51 development scenes only — test set never seen during selection.
Best dev threshold: **t=0.35** (dev F1=0.547). Applied to test set: **micro F1=0.523**.

In [ ]:
# ── Cell 3: Dev-optimised threshold (honest baseline) ──
# Already completed. Results saved to results/ndwi/ndwi_dev_threshold.json
# Dev sweep: 23 values (-0.3 to 0.80) on 12,657 development chips.
# Best dev threshold: t=0.35 (dev F1=0.547)
# Test evaluation at t=0.35: micro F1=0.523, P=0.376, R=0.859

ndwi_dev_json = NDWI_DIR / 'ndwi_dev_threshold.json'
if ndwi_dev_json.exists():
    result = json.load(open(ndwi_dev_json))
    print(f'Dev-optimised threshold: t={result["best_threshold_from_dev"]}')
    print(f'Dev F1 at t={result["best_threshold_from_dev"]}: {result["dev_f1"]:.4f}')
    print(f'\nTest set results at t={result["best_threshold_from_dev"]}:')
    print(f'  Micro F1:  {result["test_f1"]:.4f}')
    print(f'  Precision: {result["test_precision"]:.4f}')
    print(f'  Recall:    {result["test_recall"]:.4f}')
else:
    print('Results file not found')

Dev-optimised threshold: t=0.35
Dev F1 at t=0.35: 0.5471

Test set results at t=0.35:
  Micro F1:  0.5231
  Precision: 0.3760
  Recall:    0.8589

## Summary

In [ ]:
# ── Cell 4: Summary ──
dev_t = result["best_threshold_from_dev"]
ub_t  = best_row["threshold"]

print('NDWI Evaluation Summary')
print('=' * 50)
print(f'{"Method":<35} {"F1":>6} {"P":>6} {"R":>6}')
print('-' * 50)
print(f'{f"NDWI t={dev_t} (dev-optimised, honest)":<35} '
      f'{result["test_f1"]:>6.3f} {result["test_precision"]:>6.3f} {result["test_recall"]:>6.3f}')
print(f'{f"NDWI t={ub_t} (test-optimised, upper bound)":<35} '
      f'{best_row["micro_f1"]:>6.3f} {best_row["micro_precision"]:>6.3f} {best_row["micro_recall"]:>6.3f}')
print('=' * 50)
print(f'Primary result for paper: t={dev_t} (honest baseline)')
print(f'Upper bound reported for context: t={ub_t}')


NDWI Evaluation Summary
Method                               F1      P      R
--------------------------------------------------
NDWI t=0.35 (dev-optimised, honest) 0.523  0.376  0.859
NDWI t=0.70 (test-optimised, upper) 0.601  0.571  0.635
Primary result for paper: t=0.35 (honest baseline)
Upper bound reported for context: t=0.70